# Hotel cancellation risk: ML evidence

This compact notebook reads the frozen version 2.0 artifacts. The reproducible training entry point is `scripts/train_model.py`; this notebook presents the resulting evidence without hiding model logic inside notebook state.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
metadata = json.loads((ROOT / 'artifacts/model_metadata.json').read_text())
comparison = pd.read_csv(ROOT / 'reports/tables/model-comparison.csv')
test_metrics = pd.read_csv(ROOT / 'reports/tables/test-metrics.csv').iloc[0]
top_features = pd.read_csv(ROOT / 'reports/tables/top-features.csv')
monthly = pd.read_csv(ROOT / 'reports/tables/monthly-stability.csv')
assert metadata['model_version'] == '2.0.0'

## Prediction question and split design

The model predicts cancellation risk at **booking creation**. Exact duplicates are removed before chronological splitting. Model choice uses expanding-window validation; the later validation period is reserved for calibration and threshold choice; 2017 remains untouched until final evaluation.

In [ ]:
split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'period': [
        ' to '.join(metadata['split']['train_period']),
        ' to '.join(metadata['split']['validation_period']),
        ' to '.join(metadata['split']['test_period']),
    ],
    'rows': [
        metadata['split']['train_rows'],
        metadata['split']['validation_rows'],
        metadata['split']['test_rows'],
    ],
})
display(split_summary)
print(f"Removed exact duplicates: {metadata['exact_duplicate_rows_removed']:,}")

## Temporal model selection

![Temporal model comparison](reports/figures/model-comparison.png)

In [ ]:
columns = ['model', 'temporal_pr_auc_mean', 'temporal_pr_auc_std', 'temporal_roc_auc_mean', 'temporal_brier_mean']
display(comparison[columns].round(4))
assert comparison.iloc[0]['model'] == metadata['selection']['selected_model']

## Untouched future-period evaluation

The lower score than the superseded random-split analysis is expected: duplicates and future information no longer inflate performance.

![ROC and precision-recall curves](reports/figures/test-curves.png)

In [ ]:
evidence = pd.Series({
    'ROC-AUC': test_metrics['roc_auc'],
    'PR-AUC': test_metrics['pr_auc'],
    'Brier (raw)': test_metrics['brier_raw'],
    'Brier (calibrated)': test_metrics['brier_calibrated'],
    'Recall at selected threshold': test_metrics['recall'],
    'Precision at selected threshold': test_metrics['precision'],
})
display(evidence.to_frame('value').round(4))
assert test_metrics['brier_calibrated'] < test_metrics['brier_raw']

## Explainability and temporal stability

SHAP contributions describe model influence, not causality. Monthly performance shows why a live model needs drift monitoring.

![Global feature influence](reports/figures/feature-importance.png)

![Monthly temporal stability](reports/figures/temporal-stability.png)

In [ ]:
readable_features = top_features.head(10).copy()
readable_features['feature'] = (readable_features['feature']
    .str.replace(r'^(categorical|numeric)__', '', regex=True)
    .str.replace('_', ' '))
display(readable_features[['feature', 'mean_abs_shap']].round(4))
display(monthly.round(4))

## Frozen scoring path

The final check loads the persisted preprocessor, XGBoost model, and portable JSON calibrator and scores raw booking rows through the same feature contract used in training.

In [ ]:
from hotel_cancellation.scoring import load_artifacts, score_bookings

raw = pd.read_csv(ROOT / 'data/2022_hotel_bookings.csv').head(5)
scores = score_bookings(raw, load_artifacts())
display(scores.to_frame())
assert scores.between(0, 1).all()